# Demeter Interactive Testing Environment
This notebook allows you to interactively load models, run predictions, and test the inference pipeline and status engine on the fly without needing to spin up the Flask web server.

In [ ]:
import os
import sys
import json
import random
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

# --- DYNAMIC PROJECT ROOT RESOLUTION ---
PROJECT_ROOT = Path(os.getcwd()).parent if Path(os.getcwd()).name == 'notebooks' else Path(os.getcwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

try:
    from src.core.inference_engine import load_models, diagnose_plant_disease, predict_growth_milestone, generate_complete_diagnosis
except ModuleNotFoundError:
    from inference_engine import load_models, diagnose_plant_disease, predict_growth_milestone, generate_complete_diagnosis

# --- CONFIGURATION LOADING ---
with open(PROJECT_ROOT / 'config.json', 'r') as f:
    config = json.load(f)

plantvillage_dir = str(PROJECT_ROOT / config['paths']['plantvillage_dir'])
plantvillage_cnn_path = str(PROJECT_ROOT / config['paths']['plantvillage_cnn_model_path'])
danforth_rf_path = str(PROJECT_ROOT / config['paths']['danforth_rf_model_path'])

## 1. Load AI Models

In [ ]:
print("Loading models...")
cnn_model, rf_model = load_models(plantvillage_cnn_path, danforth_rf_path)
print("Models loaded successfully!")

# Load class directories for CNN (Disease classes)
class_dirs = sorted([d for d in os.listdir(plantvillage_dir) 
                   if os.path.isdir(os.path.join(plantvillage_dir, d))])

## 2. Test Image Classification (CNN)

In [ ]:
# Select a random disease class and image from the local dataset
random_class = random.choice(class_dirs)
class_path = os.path.join(plantvillage_dir, random_class)
images = [f for f in os.listdir(class_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
random_image = random.choice(images)
image_path = os.path.join(class_path, random_image)

# Display image
img = Image.open(image_path)
plt.imshow(img)
plt.axis('off')
plt.title(f"True Class: {random_class}")
plt.show()

# Run CNN prediction
disease_diagnosis = diagnose_plant_disease(image_path, cnn_model, class_dirs)
print(f"Detected Disease: {disease_diagnosis['Detected_Disease']}")
print(f"Confidence: {disease_diagnosis['Disease_Confidence']:.2%}")

## 3. Test Growth Prediction (Random Forest)

In [ ]:
# Sample environmental data (You can tweak these values to see how the model reacts)
env_data = {
    "Soil_Type": 1,
    "Sunlight_Hours": 6.5,
    "Water_Frequency": 2,
    "Fertilizer_Type": 1,
    "Temperature": 24.0,
    "Humidity": 60.0
}

growth_pred = predict_growth_milestone(env_data, rf_model)
print("Environmental Conditions:", env_data)
print(f"Predicted Growth Milestone: {growth_pred['Predicted_Growth_Milestone']:.4f}")

## 4. Generate Complete Diagnosis
This simulates what the backend API sends to the dashboard frontend, generating health scores, actions, and 7-day trajectories based on the outputs generated above.

In [ ]:
# Generate the full diagnosis payload (combines CNN, RF, and the Status Engine)
complete_diagnosis = generate_complete_diagnosis(
    image_path=image_path,
    detected_disease=disease_diagnosis['Detected_Disease'],
    disease_confidence=disease_diagnosis['Disease_Confidence'],
    all_predictions=disease_diagnosis['All_Predictions'],
    predicted_growth=growth_pred['Predicted_Growth_Milestone'],
    temperature=env_data['Temperature'],
    soil_moisture=45.0, # Tweak this to see recommendation changes
    sunlight_hours=env_data['Sunlight_Hours'],
    humidity=env_data['Humidity']
)

print("\n--- COMPLETE DASHBOARD PAYLOAD ---\n")
print(json.dumps(complete_diagnosis, indent=2, default=str))